# 01 pamoka – Įvadas į DI agentus

Sveiki atvykę į pirmąją **DI agentų pradedantiesiems** kurso pamoką!

**DI agentas** yra programa, kuri naudoja didelį kalbos modelį (LLM) kaip savo mąstymo variklį ir gali imtis *veiksmų* realiame pasaulyje – kviesti API, užklausinėti duomenų bazes arba vykdyti kodą – siekdama įgyvendinti tikslą naudotojo naudai.

Šiame užrašų knygelėje sukursite savo pirmąjį agentą: **Kelionių agentą**, kuris siūlo atostogų vietas. Kelyje išmoksite:

1. Prisijungti prie Azure AI Foundry Agent Service naudojant **Microsoft Agent Framework**.
2. Suteikti agentui **įrankį** – paprastą Python funkciją, kurią jis galės kviesti.
3. Paleisti agentą ir patikrinti jo atsakymą.
4. Srautu perduoti agento atsako žodžius po vieną (tokeną).


## Sąranka

Prieš paleisdami šį užrašų knygelę, įsitikinkite, kad turite:

1. **Azure AI Foundry projektą** su įdiegtu pokalbių modeliu (pvz., `gpt-4o-mini`).
2. **Prisijungėte per Azure CLI** — terminale paleiskite komandą `az login`.
3. **Nustatėte reikiamus aplinkos kintamuosius:**
   - `AZURE_AI_PROJECT_ENDPOINT` — jūsų Azure AI Foundry projekto galinis taškas.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — įdiegto modelio pavadinimas.

Toliau esanti ląstelė įdiegia būtinas Python paketas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pirmojo agento kūrimas

Agentui reikia dviejų dalykų:

- **Instrukcijų**, kurios nurodo *kas jis yra* ir *kaip elgtis* (sistemos užklausa).
- **Įrankių** — Python funkcijų, pažymėtų `@tool`, kurias agentas gali kviesti informacijos gavimui arba veiksmams atlikti.

Žemiau apibrėžiame paprastą įrankį, kuris pateikia populiarių atostogų krypčių sąrašą. Agentas naudos šį įrankį, kai vartotojas prašys kelionės rekomendacijų.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Srautinis atsakymas

Dėl interaktyvesnės patirties galite **srautu** gauti agento atsakymą. Vietoj to, kad lauktumėte viso atsakymo, agentas pateikia teksto dalis jas generuodamas. Tai ypač naudinga pokalbių sąsajose, kur norite rodyti rezultatą realiuoju laiku.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Santrauka

Šioje pamokoje sužinojote, kaip:

- **Sukurti tiekėją**, kuris jungiasi prie Azure AI Foundry Agent Service per `AzureAIProjectAgentProvider`.
- **Apibrėžti įrankį** naudojant `@tool` dekoratorių, kad agentas galėtų kviesti jūsų Python funkcijas.
- **Paleisti agentą** su vartotojo žinute ir atspausdinti jo atsakymą.
- **Transliuoti atsakymus** realiu laiku.

Kitoje pamokoje gilinsimės į agentinius karkasus ir mokysimės, kaip suteikti agentams galingesnius įrankius bei daugiažingsnį samprotavimą.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:  
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, atkreipkite dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Kritinei informacijai rekomenduojamas profesionalus žmogaus atliekamas vertimas. Mes neatsakome už bet kokius nesusipratimus ar klaidingas aiškinimas, kilusius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
